<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px 30px 20px; border-radius: 10px; color: white; margin-bottom: 10px;">

# NexAU Cookbook — 构建数据库 Agent

输入 **"海淀区注册资本最高的企业是哪家？"**，从数据库中得到准确答案。


</div>

### 核心概念

| 术语 | 含义 |
|:---|:---|
| **LLM**（大语言模型） | 理解你问题的 AI 模型，如 GPT-4、DeepSeek |
| **Tool**（工具） | Agent 可调用的函数，如 `execute_sql` |
| **Skill**（技能） | 教 Agent 理解数据的知识文件（SKILL.md） |
| **Agent**（智能体） | 自主行动的 LLM：调用工具 → 解读结果 → 回答你 |

### 教程路线图

| 章节 | 内容 | 你会得到 |
|:---:|:---|:---|
| **1** | 创建企业数据库 | 一个 7 张表、50 家企业的真实场景数据库 |
| **2** | 构建 `execute_sql` 工具 | 三层安全的 SQL 执行器 |
| **3** | 编写 SKILL.md | 让 Agent 正确回答的领域知识 |
| **4** | 自动生成 Skill | 30 张表也能快速起步 |
| **5** | System Prompt | Agent 的操作手册 |
| **6** | 总结 | 核心要点回顾 |

---

### 准备工作

导入标准库模块：

In [ ]:
from __future__ import annotations

import json
import os
import re
import sqlite3
import time
from pathlib import Path
from typing import Any

---

### 1. 创建企业数据库

我们用一个**企业智能数据库**场景，包含 7 张表、50 家企业：

<div style="background:#f8f9fa; padding:15px; border-radius:8px; font-family:monospace; font-size:14px;">
enterprise_basic（企业基本信息）◄──┐<br>
enterprise_contact（联系人）&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;│<br>
enterprise_financing（融资上市）&nbsp;&nbsp;├── credit_code（统一社会信用代码）<br>
enterprise_product（产品与专利）&nbsp;&nbsp;│<br>
industry_enterprise（行业映射）&nbsp;&nbsp;&nbsp;┘<br>
<br>
industry（行业链节点）◄── industry_enterprise.industry_id<br>
<br>
users（平台用户）—— 独立表，与企业无关
</div>

<div class="alert alert-warning" style="margin-top:15px;">
<strong>⚠️ 注意：</strong>这个数据库和真实场景一样有"坑"：
<ul>
<li><code>register_capital</code>（注册资本）是 TEXT 类型 — <code>ORDER BY</code> 按字母序排列</li>
<li><code>daily_capacity</code>（日产能）也是 TEXT — 直接比较会出错</li>
<li><code>zhuanjingtexin_level</code>（专精特新等级）有隐式高低之分</li>
<li><code>users</code> 表容易被误用 — "用户"在这个场景通常指企业，不是平台账号</li>
</ul>
</div>

In [ ]:
DB_PATH = Path("enterprise.sqlite")

# 如果已存在则删除重建
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

# -- enterprise_basic（企业基本信息，37 列）--------------------------
cur.execute('''
CREATE TABLE enterprise_basic (
    id INTEGER PRIMARY KEY,
    credit_code TEXT NOT NULL,
    enterprise_name TEXT,
    declaration_year TEXT,
    data_batch TEXT,
    sequence_number INTEGER,
    register_district TEXT,
    jurisdiction_district TEXT,
    street TEXT,
    register_address TEXT,
    correspondence_address TEXT,
    postal_code TEXT,
    register_date TEXT,
    register_capital TEXT,
    register_capital_currency TEXT,
    enterprise_scale TEXT,
    enterprise_type TEXT,
    foreign_capital_ratio REAL,
    industry_level1 TEXT,
    industry_level2 TEXT,
    industry_level3 TEXT,
    industry_level4 TEXT,
    main_product_service TEXT,
    main_product_category TEXT,
    market_years INTEGER,
    enterprise_introduction TEXT,
    website TEXT,
    declaration_type TEXT,
    zhuanjingtexin_level TEXT,
    financial_outlier_analysis INTEGER,
    municipal_high_level_enterprise INTEGER,
    data_source TEXT,
    created_at TEXT,
    updated_at TEXT,
    unicorn_category TEXT,
    unicorn_year INTEGER,
    legal_entity_data TEXT
)
''')

# -- enterprise_contact（联系人）-------------------------------------
cur.execute('''
CREATE TABLE enterprise_contact (
    id INTEGER PRIMARY KEY,
    credit_code TEXT NOT NULL,
    legal_person_name TEXT,
    legal_person_position TEXT,
    legal_person_phone TEXT,
    legal_person_mobile TEXT,
    manager_name TEXT,
    manager_position TEXT,
    manager_phone TEXT,
    manager_mobile TEXT,
    contact_name TEXT,
    contact_position TEXT,
    contact_phone TEXT,
    contact_mobile TEXT,
    fax TEXT,
    email TEXT,
    controlling_shareholder TEXT,
    actual_controller TEXT,
    actual_controller_nationality TEXT,
    data_source TEXT,
    created_at TEXT,
    updated_at TEXT
)
''')

# -- enterprise_financing（融资与上市）-------------------------------
cur.execute('''
CREATE TABLE enterprise_financing (
    id INTEGER PRIMARY KEY,
    credit_code TEXT NOT NULL,
    applied_bank_loan INTEGER,
    credit_satisfaction_rate REAL,
    loan_purpose TEXT,
    next_financing_plan TEXT,
    next_financing_demand REAL,
    next_financing_method TEXT,
    recent_equity_financing REAL,
    recent_valuation REAL,
    listing_status TEXT,
    stock_code TEXT,
    listing_progress TEXT,
    planned_listing_location TEXT,
    overseas_listing TEXT,
    data_source TEXT,
    created_at TEXT,
    updated_at TEXT
)
''')

# -- enterprise_product（产品与知识产权）-----------------------------
cur.execute('''
CREATE TABLE enterprise_product (
    id INTEGER PRIMARY KEY,
    credit_code TEXT NOT NULL,
    product_index INTEGER,
    product_name TEXT,
    product_revenue REAL,
    daily_capacity TEXT,
    capacity_unit TEXT,
    ip_name_1 TEXT,
    ip_name_2 TEXT,
    ip_name_3 TEXT,
    data_source TEXT,
    created_at TEXT,
    updated_at TEXT
)
''')

# -- industry（行业链节点）-------------------------------------------
cur.execute('''
CREATE TABLE industry (
    id INTEGER PRIMARY KEY,
    chain_id INTEGER NOT NULL,
    parent_id INTEGER,
    name TEXT NOT NULL,
    description TEXT,
    path TEXT NOT NULL,
    depth INTEGER NOT NULL,
    sort_order INTEGER,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    chain_position TEXT,
    icon TEXT
)
''')

# -- industry_enterprise（企业-行业映射）-----------------------------
cur.execute('''
CREATE TABLE industry_enterprise (
    industry_id INTEGER,
    credit_code TEXT,
    created_at TEXT NOT NULL,
    chain_id INTEGER NOT NULL,
    industry_path TEXT NOT NULL,
    PRIMARY KEY (industry_id, credit_code)
)
''')

# -- users（平台用户，与企业无关）------------------------------------
cur.execute('''
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    sso_user_id TEXT NOT NULL,
    display_name TEXT,
    email TEXT,
    role TEXT NOT NULL,
    created_at TEXT,
    updated_at TEXT,
    sso_raw_data TEXT,
    password_hash TEXT,
    username TEXT
)
''')

# === 插入示例数据 ===
import random
random.seed(42)

districts = ['海淀区', '朝阳区', '黄浦区', '南山区', '余杭区']
scales = ['微型', '小型', '中型', '大型']
types = ['民营', '国有', '合资', '外资']
industries_l1 = ['制造业', '信息传输、软件和信息技术服务业', '科学研究和技术服务业', '金融业']
ztx_levels = [None, None, '专精特新中小企业', '专精特新潜在"小巨人"企业', '专精特新"小巨人"企业']
loan_purposes = ['流动资金', '设备采购', '研发投入', '扩大产能']
financing_plans = ['暂无', '12个月内', '24个月内']
financing_methods = ['股权融资', '债权融资', '可转债']
listing_statuses = ['未上市', '未上市', '未上市', '新三板', '已上市', '拟上市']
listing_locations = ['无', '上交所', '深交所', '北交所', '港交所']

for i in range(1, 51):
    cc = f'MOCKCREDIT{i:010d}'
    cap = str(random.choice([100, 500, 1000, 3000, 5000, 8000, 10000, 50000]))
    cur.execute(
        "INSERT INTO enterprise_basic (id, credit_code, enterprise_name, "
        "register_district, register_capital, register_capital_currency, "
        "enterprise_scale, enterprise_type, industry_level1, "
        "zhuanjingtexin_level, register_date, created_at, updated_at) "
        "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)",
        (i, cc, f'测试企业_{i}', random.choice(districts), cap, '人民币',
         random.choice(scales), random.choice(types),
         random.choice(industries_l1), random.choice(ztx_levels),
         f'20{random.randint(10,24)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}',
         '2025-01-01', '2025-01-01')
    )

    # contact
    cur.execute(
        "INSERT INTO enterprise_contact (id, credit_code, legal_person_name, "
        "legal_person_position, email, created_at, updated_at) VALUES (?,?,?,?,?,?,?)",
        (i, cc, f'REDACTED_{i}', random.choice(['董事长', '总经理', '董事长兼首席执行官']),
         f'user{i}@example.com', '2025-01-01', '2025-01-01')
    )

    # financing
    ls = random.choice(listing_statuses)
    cur.execute(
        "INSERT INTO enterprise_financing (id, credit_code, applied_bank_loan, "
        "credit_satisfaction_rate, loan_purpose, listing_status, stock_code, "
        "planned_listing_location, next_financing_demand, recent_valuation, "
        "created_at, updated_at) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",
        (i, cc, random.randint(0,1), round(random.uniform(0.3,1.0),2),
         random.choice(loan_purposes), ls,
         f'{random.randint(600000,699999)}' if ls == '已上市' else '',
         random.choice(listing_locations) if ls != '未上市' else '无',
         round(random.uniform(500, 50000), 0),
         round(random.uniform(5000, 500000), 0),
         '2025-01-01', '2025-01-01')
    )

    # products (1-3 per enterprise)
    n_products = random.randint(1, 3)
    for j in range(1, n_products + 1):
        cur.execute(
            "INSERT INTO enterprise_product (id, credit_code, product_index, "
            "product_name, product_revenue, daily_capacity, capacity_unit, "
            "created_at, updated_at) VALUES (?,?,?,?,?,?,?,?,?)",
            ((i-1)*3+j, cc, j, f'产品_{i}_{j}',
             round(random.uniform(100, 20000), 2),
             str(random.randint(10, 500)),
             random.choice(['件/日', '吨/日', '台/日', '套/日']),
             '2025-01-01', '2025-01-01')
        )

# Industry chain (AI chain, chain_id=45)
chain_nodes = [
    (1, 45, None, 'AI 产业链', '', '[]', 0, 1, 'AI产业链总览', None),
    (2, 45, 1, '上游（基础要素与基础设施）', '', '[1]', 1, 1, '基础设施与核心技术', 'up'),
    (3, 45, 1, '中游（技术与平台）', '', '[1]', 1, 2, '技术平台与算法', 'mid'),
    (4, 45, 1, '下游（应用）', '', '[1]', 1, 3, '行业应用场景', 'down'),
    (5, 45, 2, '算力 — GPU 集群', '', '[1,2]', 2, 1, 'GPU与高性能计算', None),
    (6, 45, 2, '数据 — 标注与治理', '', '[1,2]', 2, 2, '数据标注、清洗与治理', None),
    (7, 45, 3, '大模型平台', '', '[1,3]', 2, 1, '大语言模型训练与部署', None),
    (8, 45, 3, 'AI 中间件', '', '[1,3]', 2, 2, 'AI开发框架与工具链', None),
    (9, 45, 4, '智能制造', '', '[1,4]', 2, 1, '工业AI应用', None),
    (10, 45, 4, '智慧医疗', '', '[1,4]', 2, 2, '医疗AI应用', None),
]
for node in chain_nodes:
    cur.execute(
        "INSERT INTO industry (id, chain_id, parent_id, name, description, path, "
        "depth, sort_order, created_at, updated_at, chain_position) "
        "VALUES (?,?,?,?,?,?,?,?,?,?,?)",
        (node[0], node[1], node[2], node[3], node[8], node[5], node[6],
         node[7], '2025-01-01', '2025-01-01', node[9])
    )

# Map some enterprises to industry chain nodes
for i in range(1, 21):
    cc = f'MOCKCREDIT{i:010d}'
    node_id = random.choice([2, 3, 4, 5, 6, 7, 8, 9, 10])
    node = [n for n in chain_nodes if n[0] == node_id][0]
    cur.execute(
        "INSERT OR IGNORE INTO industry_enterprise "
        "(industry_id, credit_code, created_at, chain_id, industry_path) "
        "VALUES (?,?,?,?,?)",
        (node_id, cc, '2025-01-01', 45, node[5])
    )

# Platform users
for i in range(1, 6):
    cur.execute(
        "INSERT INTO users (id, sso_user_id, display_name, email, role, "
        "created_at, updated_at) VALUES (?,?,?,?,?,?,?)",
        (i, f'SSO_{i}', f'REDACTED_{i}', f'user{i}@example.com',
         'admin' if i <= 2 else 'user', '2025-01-01', '2025-01-01')
    )

conn.commit()
conn.close()

# 统计
conn = sqlite3.connect(str(DB_PATH))
tables = ['enterprise_basic', 'enterprise_contact', 'enterprise_financing',
          'enterprise_product', 'industry', 'industry_enterprise', 'users']
print(f"Database created: {DB_PATH}")
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")
conn.close()

看一下核心表的数据：

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

for table in ["enterprise_basic", "enterprise_financing", "enterprise_product", "industry"]:
    rows = conn.execute(f"SELECT * FROM {table} LIMIT 2").fetchall()
    cols = [desc[0] for desc in conn.execute(f"SELECT * FROM {table} LIMIT 1").description]
    print(f"\n-- {table}（前 2 行，共 {len(cols)} 列） --")
    for r in rows:
        d = dict(r)
        # 只显示有值的列
        shown = {k: v for k, v in d.items() if v is not None and v != ''}
        print(json.dumps(shown, ensure_ascii=False, indent=2)[:300])

conn.close()

---

### 2. 构建 `execute_sql` 工具

这是 Agent 与数据库对话的**唯一通道**。

<div style="background:#e8f4fd; border-left:4px solid #1976d2; padding:12px 15px; border-radius:0 8px 8px 0; margin:10px 0;">
💡 <strong>打个比方：</strong><code>execute_sql</code> 就像银行的防弹玻璃窗口 — 客户（LLM）可以问问题、拿答案，但绝对伸不进去动金库。
</div>

**三层安全机制：**

<table style="width:100%; border-collapse: separate; border-spacing: 0 6px;">
<tr><td style="background:#c8e6c9; padding:10px 15px; border-radius:8px;">
🛡️ <strong>第 1 层：关键字检查</strong> — SQL 是 SELECT 开头吗？
</td></tr>
<tr><td style="background:#fff9c4; padding:10px 15px; border-radius:8px;">
🛡️ <strong>第 2 层：注释剥离</strong> — 有没有人把 DELETE 藏在注释后面？
</td></tr>
<tr><td style="background:#ffcdd2; padding:10px 15px; border-radius:8px;">
🛡️ <strong>第 3 层：只读连接</strong> — 即使前两层漏了，SQLite 也拒绝写入
</td></tr>
</table>

**结构化输出：** 返回 JSON 字典（`status`、`data`、`warnings`），查询返回 0 行时自动提示模型检查条件 — 这就是 Agent **自我纠错**的机制。

#### 2.1 实现代码

In [ ]:
MAX_ROWS = 10
MAX_OUTPUT_LENGTH = 50_000
DEFAULT_TIMEOUT = 30

_DANGEROUS_KEYWORDS = (
    "DROP", "TRUNCATE", "DELETE", "ALTER", "CREATE",
    "INSERT", "UPDATE", "REPLACE", "ATTACH", "DETACH",
    "PRAGMA", "VACUUM", "REINDEX", "GRANT", "REVOKE",
)


def _strip_sql_comments(sql: str) -> str:
    no_line = re.sub(r"--[^\n]*", "", sql)
    no_block = re.sub(r"/\*.*?\*/", "", no_line, flags=re.DOTALL)
    return no_block


def execute_sql(
    sql: str,
    db_path: str | Path = DB_PATH,
    timeout: int = DEFAULT_TIMEOUT,
    max_rows: int = MAX_ROWS,
) -> dict[str, Any]:
    start = time.time()

    if not sql or not sql.strip():
        return {"status": "error", "error": "SQL query cannot be empty",
                "duration_ms": int((time.time() - start) * 1000)}

    cleaned_upper = _strip_sql_comments(sql).strip().upper()
    for kw in _DANGEROUS_KEYWORDS:
        if re.match(rf"^{kw}(?:\s|$)", cleaned_upper):
            return {"status": "error",
                    "error": f"Only SELECT queries allowed. Found: {kw}",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}

    if not re.match(r"^(SELECT|WITH)\b", cleaned_upper):
        return {"status": "error",
                "error": "Only SELECT or WITH ... SELECT queries allowed.",
                "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}

    connection = None
    try:
        uri = f"file:{db_path}?mode=ro"
        connection = sqlite3.connect(uri, uri=True)
        connection.row_factory = sqlite3.Row

        deadline = time.time() + timeout
        connection.set_progress_handler(
            lambda: 1 if time.time() > deadline else 0, 1000
        )

        cursor = connection.execute(sql)
        col_names = [d[0] for d in cursor.description] if cursor.description else []

        batch = cursor.fetchmany(max_rows + 1)
        truncated = len(batch) > max_rows
        rows_to_convert = batch[:max_rows]
        total_rows = max_rows + 1 if truncated else len(batch)

        data = [dict(r) for r in rows_to_convert]
        duration_ms = int((time.time() - start) * 1000)

        warnings: list[str] = []
        if total_rows == 0:
            warnings.append(
                "Query returned 0 rows. Check: table name, column names, "
                "WHERE conditions, data availability."
            )
        if truncated:
            warnings.append(
                f"Showing {max_rows} rows (more available). "
                f"Add WHERE clauses or LIMIT to narrow results."
            )

        result: dict[str, Any] = {
            "status": "success",
            "sql": sql,
            "columns": col_names,
            "data": data,
            "row_count": len(data),
            "total_rows": total_rows,
            "truncated": truncated,
            "duration_ms": duration_ms,
        }

        if len(json.dumps(result, ensure_ascii=False)) > MAX_OUTPUT_LENGTH:
            while (
                len(data) > 1
                and len(json.dumps(result, ensure_ascii=False)) > MAX_OUTPUT_LENGTH
            ):
                data = data[:-1]
                result["data"] = data
                result["row_count"] = len(data)
                result["truncated"] = True
            if not any("length limit" in w for w in warnings):
                warnings.append(
                    "Results truncated due to output length limit. "
                    "Select fewer columns or add WHERE clauses."
                )

        if warnings:
            result["warnings"] = warnings
        return result

    except sqlite3.OperationalError as e:
        msg = str(e)
        if "interrupted" in msg.lower():
            return {"status": "timeout",
                    "error": f"Query timed out after {timeout}s",
                    "sql": sql,
                    "duration_ms": int((time.time() - start) * 1000)}
        return {"status": "error", "error": msg, "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    except Exception as e:
        return {"status": "error", "error": str(e), "sql": sql,
                "duration_ms": int((time.time() - start) * 1000)}
    finally:
        if connection:
            connection.close()

#### 2.2 测试工具

验证一下正常查询和安全拦截：

In [ ]:
# 正常查询
result = execute_sql("SELECT enterprise_name, register_capital, enterprise_scale FROM enterprise_basic LIMIT 3")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# 空结果会触发 warning，提示模型检查条件
result = execute_sql("SELECT * FROM enterprise_basic WHERE register_district = '不存在的区'")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# 安全测试：拦截 DROP — 第 1 层拦住
result = execute_sql("DROP TABLE enterprise_basic")
print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
# 安全测试：注释绕过 — 第 2 层剥除注释后，第 1 层拦住 DELETE
result = execute_sql("-- just a comment\nDELETE FROM enterprise_basic")
print(json.dumps(result, ensure_ascii=False, indent=2))

#### 2.3 TEXT 当数字的陷阱

<div class="alert alert-danger">
<strong>🐛 头号陷阱：</strong>当 <code>register_capital</code>（注册资本）以 TEXT 存储时，<code>ORDER BY register_capital</code> 按<strong>字母序</strong>排列 — <code>"8000" > "50000"</code>，因为 <code>"8" > "5"</code>。
<br><br>
解决方法：<code>CAST(register_capital AS REAL)</code> — 排序前把 TEXT 转成数字。但模型不会自动知道这一点 — 这就是 Skill 的用武之地（第 3 节）。
</div>

In [ ]:
# 错误：TEXT 排序 — "8000" 排在 "50000" 前面（字母序）
result = execute_sql("SELECT enterprise_name, register_capital FROM enterprise_basic ORDER BY register_capital DESC LIMIT 5")
print("TEXT 排序（错误）：")
for row in result["data"]:
    print(f"  {row['enterprise_name']:15s} {row['register_capital']} 万元")

print()

# 正确：CAST 转为数字后排序
result = execute_sql("SELECT enterprise_name, CAST(register_capital AS REAL) AS capital FROM enterprise_basic ORDER BY capital DESC LIMIT 5")
print("数值排序（正确）：")
for row in result["data"]:
    print(f"  {row['enterprise_name']:15s} {row['capital']} 万元")

---

### 3. 编写 SKILL.md — 提升问答准确率的关键

基础 Agent 经常答错——不是因为模型"不会写 SQL"，而是**不了解这个数据库的具体情况**。

`register_capital` 是 TEXT 类型，直接 `ORDER BY` 会按字符串排序；`zhuanjingtexin_level` 有隐式的等级高低；`users` 表和企业数据无关——这些"潜规则"模型不可能自己知道。

**Skill 就是把你对数据库的了解传递给模型的方式。**

#### 3.1 Skill 的基本结构

Skill 就是一个文件夹里放一个 `SKILL.md`。文件由两部分组成：

```markdown
---
name: enterprise_basic
description: Use this skill whenever the user asks about an enterprise's
  identity, registration, location, scale, or industry classification.
---

# enterprise_basic — 企业基本信息
（正文：Schema、示例 SQL、注意事项...）
```

- **`description`**（始终对模型可见）——告诉模型"什么时候该读取这个 Skill"
- **正文**（模型决定读取时才加载）——告诉模型"这张表具体怎么查"

接下来的重点：**正文里写什么、怎么写，才能让 Agent 答对？**

#### 3.2 第一步：审计你的数据库

编写 Skill 之前，逐表回答以下问题：

| 问题 | 目的 |
|:---|:---|
| 这张表存的是什么？一行代表什么？ | 让模型理解业务含义 |
| 哪些列的存储类型和业务含义不匹配？ | 发现类型陷阱（TEXT 存数字） |
| 哪些列只有几个固定取值？ | 列出枚举值，减少猜测 |
| 聚合时需要排除哪些数据？ | 明确业务过滤规则 |
| 哪些字段是预计算的？ | 防止模型重复计算 |
| 表之间怎么 join？ | 标注主键和外键 |
| 用户最常问什么？正确的 SQL 怎么写？ | 准备示例查询 |

**每回答一个问题，就往 SKILL.md 里写一条。**

以 `enterprise_basic` 为例：
- 一行 = 一家企业，37 列涵盖注册信息、行业分类、专精特新等级
- `register_capital` 是 TEXT 存数字 → 需要 CAST 提示
- `enterprise_scale` 只有 4 个值（微型/小型/中型/大型）→ 列出枚举
- `zhuanjingtexin_level` 有 3 个等级 + NULL → 列出并标注高低顺序
- 通过 `credit_code` 与其他 enterprise_* 表 join

#### 3.3 类型陷阱：准确率的第一杀手

第 2 节已经见过了——`register_capital` 是 TEXT，直接 `ORDER BY register_capital DESC` 结果完全错误。

**Skill 中需要在三个地方提示模型**：

**① Schema 表——紧邻列名标注**：
```
| `register_capital` | TEXT | 注册资本（万元）— **TEXT not numeric**, use `CAST(register_capital AS REAL)` |
```

**② Example queries——展示正确写法**：
```sql
SELECT enterprise_name, CAST(register_capital AS REAL) AS capital_wan
FROM enterprise_basic
WHERE register_district = '海淀区'
ORDER BY capital_wan DESC LIMIT 10;
```

**③ Gotchas——解释为什么错**：
```
- `register_capital` is TEXT — always CAST(register_capital AS REAL).
  Direct ORDER BY gives wrong results (string sort: "8000" > "50000").
```

三处信息互相加强：Schema 让模型注意到类型，Example 提供正确模板，Gotchas 解释原因。

同理，`enterprise_product.daily_capacity` 也是 TEXT 存数字，需要同样处理。

#### 3.4 业务规则：只有你知道的知识

类型陷阱看 Schema 就能发现，但**业务规则只有了解业务的人才知道**。这是 Skill 最有价值的部分。

**枚举等级**——`zhuanjingtexin_level` 的值有隐含的高低之分：
```
- zhuanjingtexin_level hierarchy:
  专精特新中小企业 < 专精特新潜在"小巨人"企业 < 专精特新"小巨人"企业
  NULL = 无专精特新认定
  "小巨人企业": WHERE zhuanjingtexin_level = '专精特新"小巨人"企业'
```
不写这条，Agent 不知道"小巨人"对应的精确值。

**上市状态过滤**——`enterprise_financing.listing_status` 有 4 个值：
```
- listing_status values: 未上市, 新三板, 已上市, 拟上市
  "已上市的企业" = WHERE listing_status = '已上市'
  "有上市计划的" = WHERE listing_status IN ('拟上市', '已上市')
```

**行业链的树结构**——`industry` 表是一棵树：
```
- depth=0: chain root, depth=1: 上游/中游/下游, depth=2: leaf nodes
- chain_position ('up'/'mid'/'down') ONLY on depth=1 nodes
- "AI 上游企业" = join industry (chain_position='up') → industry_enterprise → enterprise_basic
```

**容易混淆的表**——`users` 是平台账号，不是企业：
```
- "用户"在本数据库中通常指企业（enterprise_basic），不是平台用户（users）
- 只有明确问"平台管理员"、"登录账号"时才查 users 表
```

#### 3.5 跨表查询：引导模型正确 Join

用户问"专精特新小巨人企业中有哪些已上市？"需要 join `enterprise_basic` 和 `enterprise_financing` 两张表。

**在 description 中标注 join 关系**：
```yaml
description: >-
  Use this skill whenever the user asks about an enterprise's identity,
  registration, location, scale, or industry classification.
  Join other enterprise_* tables via credit_code.
```

**在 Schema 中标注 join key**：
```
| `credit_code` | TEXT | **Join key.** 统一社会信用代码 — shared across all enterprise_* tables. |
```

**在 Example queries 中提供 join 示例**（同时展示 join 条件和业务过滤）：
```sql
SELECT b.enterprise_name, f.listing_status, f.stock_code
FROM enterprise_basic b
JOIN enterprise_financing f ON b.credit_code = f.credit_code
WHERE b.zhuanjingtexin_level = '专精特新"小巨人"企业'
  AND f.listing_status = '已上市';
```

**三表 join 更需要引导**——"AI 上游有哪些企业"涉及 industry → industry_enterprise → enterprise_basic：
```sql
SELECT b.enterprise_name, i.name AS industry_node
FROM industry i
JOIN industry_enterprise ie ON ie.industry_id = i.id
JOIN enterprise_basic b ON b.credit_code = ie.credit_code
WHERE i.chain_id = 45 AND i.chain_position = 'up';
```

#### 3.6 Example queries 的选择策略

Example queries 是模型编写 SQL 的直接模板。

**选择原则**：
1. 覆盖最常见的查询模式（聚合、排序、分组、join）
2. 每个示例至少展示一个"坑"的正确处理（CAST、枚举值、join 路径）
3. 选择用户真的会问的问题

**每张表 2-4 个示例**——太少模型缺少模板，太多模型可能忽略。

| 好示例 | 差示例 |
|:---|:---|
| 涉及 CAST 的聚合查询 | `SELECT * FROM table LIMIT 10` |
| 带枚举值过滤的分组统计 | 不带 WHERE 的全表查询 |
| 多表 join + 业务过滤 | 单表单列查询 |

差示例的问题：模型已经会写简单查询了。**示例要展示的是模型自己不知道的东西。**

#### 3.7 description：让模型找到正确的 Skill

`description` 决定模型"是否读取这个 Skill"。它始终可见，所以写法关键：

**正面路由**——关键词要覆盖用户的不同说法：
```yaml
description: >-
  Use this skill whenever the user asks about an enterprise's financing —
  bank loans, equity rounds, valuation, listing status, planned listing
  location, or future financing demand. Join with enterprise_basic via credit_code.
```

**负面路由**——有些表容易被混淆，需要主动"劝退"：
```yaml
description: >-
  Use this skill ONLY when the user explicitly asks about platform users —
  login accounts, SSO ids, roles. This table is unrelated to the enterprise
  tables. Most questions about "用户" actually mean enterprises, not platform users.
```

| 反模式 | 问题 | 改进 |
|:---|:---|:---|
| `"企业基本信息"` | 太笼统 | `"Use when user asks about registration, location, scale, industry, 专精特新 status"` |
| `"融资表"` | 纯标签 | `"Use when user asks about bank loans, equity, valuation, listing status"` |
| 把 Schema 塞进 description | 浪费 token | description 只放路由提示，Schema 放正文 |

#### 3.8 完整示例：核心表的 SKILL.md

下面展示 `enterprise_basic` 和 `users` 两个关键 Skill，注意观察如何运用前面讲到的技巧。

In [ ]:
# ===== enterprise_basic（最核心的表）=====
enterprise_basic_skill = '''---
name: enterprise_basic
description: >-
  Use this skill whenever the user asks about an enterprise's identity,
  registration, location, scale, industry classification, or 专精特新 status.
  This is the primary table — almost every query about a company starts here.
  Join other enterprise_* tables to it via credit_code.
---

# enterprise_basic — 企业基本信息

The central registry of enterprises. One row per enterprise, keyed by `credit_code`.

## When to use

- "Where is company X registered?" / "What district?"
- "How many small enterprises are there in 海淀区?"
- "List all 专精特新小巨人 enterprises"
- "What is the registered capital of …"

For contact info go to `enterprise_contact`, for financing go to `enterprise_financing`,
for products go to `enterprise_product`.

## Schema

| Column | Type | Description |
|---|---|---|
| `id` | INTEGER PK | Internal row id |
| `credit_code` | TEXT | **Join key.** 统一社会信用代码 — shared across all enterprise_* tables |
| `enterprise_name` | TEXT | 企业名称 |
| `register_district` | TEXT | 注册地所在区 (e.g. `海淀区`, `黄浦区`, `南山区`) |
| `register_capital` | TEXT | 注册资本（万元）— **TEXT not numeric**, use `CAST(register_capital AS REAL)` |
| `enterprise_scale` | TEXT | One of `微型`, `小型`, `中型`, `大型` |
| `enterprise_type` | TEXT | One of `民营`, `国有`, `合资`, `外资` |
| `industry_level1` | TEXT | 行业一级分类 (e.g. `制造业`, `金融业`) |
| `zhuanjingtexin_level` | TEXT | 专精特新等级 — see Common values |
| ... | | (37 columns total, see full schema for details) |

## Common values

- `enterprise_scale`: `微型`, `小型`, `中型`, `大型`
- `enterprise_type`: `民营`, `国有`, `合资`, `外资`
- `zhuanjingtexin_level`: `专精特新中小企业` < `专精特新潜在"小巨人"企业` < `专精特新"小巨人"企业`

## Example queries

**Top 10 enterprises by registered capital in 海淀区:**
```sql
SELECT enterprise_name,
       CAST(register_capital AS REAL) AS capital_wan,
       enterprise_scale
FROM enterprise_basic
WHERE register_district = '海淀区'
ORDER BY capital_wan DESC LIMIT 10;
```

**Count of 专精特新 enterprises by level:**
```sql
SELECT zhuanjingtexin_level, COUNT(*) AS n
FROM enterprise_basic
WHERE zhuanjingtexin_level IS NOT NULL
GROUP BY zhuanjingtexin_level ORDER BY n DESC;
```

**Join with financing to find listed 小巨人 companies:**
```sql
SELECT b.enterprise_name, f.listing_status, f.stock_code
FROM enterprise_basic b
JOIN enterprise_financing f ON b.credit_code = f.credit_code
WHERE b.zhuanjingtexin_level = '专精特新"小巨人"企业'
  AND f.listing_status = '已上市';
```

## Gotchas

- `register_capital` is **TEXT** — cast with `CAST(register_capital AS REAL)` for sorting/comparison.
- `zhuanjingtexin_level` contains Chinese quotes — match exactly: `'专精特新"小巨人"企业'`
- `enterprise_name` in mock data looks like `测试企业_N`.
'''

# ===== users（负面路由示例）=====
users_skill = '''---
name: users
description: >-
  Use this skill ONLY when the user explicitly asks about platform users —
  login accounts, SSO ids, roles. This table is unrelated to the enterprise
  tables and should not be joined to them. Most questions about "用户" actually
  mean enterprises, not platform users — confirm with the user if ambiguous.
---

# users — 平台用户账号

System users of the data platform itself — not enterprises.

## When to use

- "How many platform admins are there?"
- "List all users with the admin role"

**Do NOT use** when the user asks about enterprises, customers, or contacts.

## Schema

| Column | Type | Description |
|---|---|---|
| `id` | INTEGER PK | Internal id |
| `role` | TEXT | One of `user`, `admin` |
| `email` | TEXT | 邮箱 |

## Example queries

```sql
SELECT role, COUNT(*) AS n FROM users GROUP BY role;
```
'''

print(f'enterprise_basic_skill: {len(enterprise_basic_skill)} chars')
print(f'users_skill:            {len(users_skill)} chars')

#### 3.9 测试与迭代

写完 Skill 后，用以下方法验证：

1. **用 "When to use" 中的问题测试**——模型是否读取了正确的 Skill？SQL 是否正确？
2. **刻意测试边界情况**：
   - TEXT 数字列的排序："注册资本最高的企业"
   - 枚举值精确匹配："专精特新小巨人企业有哪些"
   - 跨表查询："已上市的小巨人企业"
   - 三表 join："AI 产业链上游有哪些企业"
   - 负面路由："平台有多少管理员"（应走 users，不是 enterprise_basic）
3. **每次 Agent 答错，分析根因并更新 Skill**：

| Agent 的错误 | Skill 如何修改 |
|:---|:---|
| 没有读取该 Skill | description 中加入用户使用的关键词 |
| 类型处理错误 | Schema + Gotchas + Example 三处同时加提示 |
| 枚举值写错 | Common values 中加入精确值 |
| Join 写错 | Schema 标注 FK，description 加 join 说明 |
| 误用 users 表 | users 的 description 加强负面路由 |

**Skill 是活文档——每修复一个错误就加一条提示，准确率逐步提升。**

---

### 4. 自动生成 Skill

7 张表手写没问题，70 张表就需要自动化了。下面的脚本读取任意 SQLite，为每张表生成 SKILL.md 草稿：

- 表结构（列名、类型、主键、外键）
- TEXT 列实际存储数字的情况
- 枚举型列（取值较少的列）
- 外键关系（自动生成 JOIN 示例）

需要人工判断的地方标记了 `[TODO]`。

#### 4.1 辅助函数

In [ ]:
def get_tables(conn: sqlite3.Connection) -> list[str]:
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' "
        "AND name NOT LIKE 'sqlite_%' ORDER BY name"
    ).fetchall()
    return [r[0] for r in rows]


def get_table_info(conn: sqlite3.Connection, table: str) -> list[dict]:
    rows = conn.execute(f"PRAGMA table_info('{table}')").fetchall()
    return [{"name": r[1], "type": r[2] or "TEXT", "pk": bool(r[5]),
             "notnull": bool(r[3]), "default": r[4]} for r in rows]


def get_foreign_keys(conn: sqlite3.Connection, table: str) -> dict[str, str]:
    rows = conn.execute(f"PRAGMA foreign_key_list('{table}')").fetchall()
    return {r[3]: f"{r[2]}.{r[4]}" for r in rows}


def get_common_values(conn: sqlite3.Connection, table: str, col: str,
                      limit: int = 8) -> list[tuple[str, int]]:
    try:
        rows = conn.execute(
            f"SELECT [{col}], COUNT(*) AS n FROM [{table}] "
            f"WHERE [{col}] IS NOT NULL "
            f"GROUP BY [{col}] ORDER BY n DESC LIMIT ?", (limit,)
        ).fetchall()
        return [(str(r[0]), r[1]) for r in rows]
    except Exception:
        return []


def is_numeric_in_text(conn: sqlite3.Connection, table: str, col: str) -> bool:
    try:
        sample = conn.execute(
            f"SELECT [{col}] FROM [{table}] WHERE [{col}] IS NOT NULL LIMIT 20"
        ).fetchall()
        if not sample:
            return False
        numeric_count = sum(
            1 for r in sample
            if r[0] is not None and re.match(r"^-?\d+(\.\d+)?$", str(r[0]).strip())
        )
        return numeric_count / len(sample) > 0.8
    except Exception:
        return False

#### 4.2 主生成函数

In [ ]:
def generate_skill_md(conn: sqlite3.Connection, table: str) -> str:
    columns = get_table_info(conn, table)
    fk_map = get_foreign_keys(conn, table)
    row_count = conn.execute(f"SELECT COUNT(*) FROM [{table}]").fetchone()[0]

    pk_cols = [c["name"] for c in columns if c["pk"]]
    pk_str = ", ".join(f"`{c}`" for c in pk_cols) if pk_cols else "(no explicit PK)"

    fk_notes = [f"`{lc}` -> `{ref}`" for lc, ref in fk_map.items()]
    fk_hint = " Join keys: " + ", ".join(fk_notes) + "." if fk_map else ""

    schema_rows = []
    for col in columns:
        parts = []
        if col["pk"]:
            parts.append("PK")
        if col["name"] in fk_map:
            parts.append(f"FK -> `{fk_map[col['name']]}`")
        if col["notnull"] and not col["pk"]:
            parts.append("NOT NULL")
        if col["default"] is not None:
            parts.append(f"default: {col['default']}")
        desc = " | ".join(parts) if parts else ""
        schema_rows.append(f"| `{col['name']}` | {col['type']} | {desc} |")

    common_sections = []
    for col in columns:
        if col["type"].upper() not in ("TEXT",) or col["pk"]:
            continue
        vals = get_common_values(conn, table, col["name"])
        if 2 <= len(vals) <= 10:
            val_str = ", ".join(f"`{v[0]}`" for v in vals)
            common_sections.append(f"- `{col['name']}`: {val_str}")

    gotchas = []
    for col in columns:
        if col["type"].upper() == "TEXT" and is_numeric_in_text(conn, table, col["name"]):
            gotchas.append(
                f"- `{col['name']}` is **TEXT** but stores numeric values — "
                f"use `CAST({col['name']} AS REAL)` for comparisons and sorting."
            )

    select_cols = ", ".join(f"[{c['name']}]" for c in columns[:5])
    examples = f"**Browse:**\n\n```sql\nSELECT {select_cols}\nFROM [{table}]\nLIMIT 10;\n```"

    if fk_map:
        fk_col, ref = next(iter(fk_map.items()))
        ref_table, ref_col = ref.split(".")
        examples += (
            f"\n\n**Join with `{ref_table}`:**\n\n```sql\n"
            f"SELECT t.*, r.*\nFROM [{table}] t\n"
            f"JOIN [{ref_table}] r ON t.[{fk_col}] = r.[{ref_col}]\n"
            f"LIMIT 10;\n```"
        )

    nl = "\n"
    return (f"---\nname: {table}\ndescription: >-\n"
            f"  Use this skill whenever the user asks about data in the `{table}` table.{fk_hint}\n"
            f"  [TODO: Add routing keywords]\n---\n\n"
            f"# {table}\n\nRows: **{row_count}** | PK: {pk_str}\n"
            + (f"Foreign keys: {', '.join(fk_notes)}\n" if fk_notes else "")
            + f"\n## When to use\n\n- [TODO: Add 3-5 example questions]\n"
            f"\n## Schema\n\n| Column | Type | Description |\n|---|---|---|\n"
            f"{nl.join(schema_rows)}\n"
            f"\n## Common values\n\n"
            f"{nl.join(common_sections) if common_sections else '- [TODO: Add common values]'}\n"
            f"\n## Example queries\n\n{examples}\n"
            f"\n## Gotchas\n\n"
            f"{nl.join(gotchas) if gotchas else '- [TODO: Add known pitfalls]'}\n")

#### 4.3 在企业数据库上运行

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
tables = get_tables(conn)
print(f"Found {len(tables)} tables: {', '.join(tables)}\n")

for table in tables[:3]:  # 只展示前 3 张表，避免输出过长
    print("=" * 60)
    print(f"  {table}")
    print("=" * 60)
    print(generate_skill_md(conn, table))

conn.close()

看看自动生成器发现了什么：

- `enterprise_basic.register_capital` 被标记为"TEXT 存储数字" — 头号陷阱
- 枚举列（`enterprise_scale`、`enterprise_type`、`listing_status`）的取值被列出
- `[TODO]` 标记了需要你补充业务上下文的地方——比如专精特新等级的高低关系、跨表 join 路径

**自动生成是起点，人工补充业务知识才是关键。**

---

### 5. System Prompt（系统提示词）

**系统提示词**是 Agent 的操作手册，定义了它的工作方式：

```
规划 → 读 Skill → 写 SQL → 执行 → 反思 → 回答
```

> **Skill vs System Prompt 的分工：** Skill 存放每张表的详细知识（动态加载），System Prompt 定义工作流和全局约束（始终可见）。

In [ ]:
SYSTEM_PROMPT = '''
You are an enterprise data agent for the North Nova enterprise intelligence
database — a SQLite mirror of seven core tables describing Chinese enterprises,
their contacts, financing status, products, and the industry chains they belong to.

## Database

- Engine: SQLite (read-only via `execute_sql`)
- Tables: `enterprise_basic`, `enterprise_contact`, `enterprise_financing`,
  `enterprise_product`, `industry`, `industry_enterprise`, `users`
- Primary join key across enterprise_* tables: `credit_code`

## Table Knowledge (Skills)

Detailed schema, common values, and example queries for each table are
provided as Skills. ALWAYS read the relevant Skill before writing a query.

### Key gotchas (always visible)

- `enterprise_basic.register_capital` is TEXT — use CAST(register_capital AS REAL)
- `enterprise_product.daily_capacity` is TEXT — use CAST(daily_capacity AS REAL)
- `zhuanjingtexin_level` values: 专精特新中小企业 < 专精特新潜在"小巨人"企业 < 专精特新"小巨人"企业
- `users` table = platform accounts, NOT enterprises. "用户" usually means enterprise.
- `industry.chain_position` ('up'/'mid'/'down') only on depth=1 nodes

## Workflow

1. **Plan.** Identify which tables you need.
2. **Read Skills.** For every table you will touch, read its Skill first.
3. **Write the SQL.** SQLite syntax. Always LIMIT. Prefer explicit columns.
4. **Execute.** Call `execute_sql`.
5. **Reflect.** If 0 rows or warnings, re-read the Skill and try again.
6. **Answer** in the user\'s language. Be concise. Show key data + SQL.

## Constraints

- Only SELECT queries work — the tool rejects everything else.
- Don\'t hallucinate columns. If unsure, query the schema first.
- Mock data: names like 测试企业_N, credit codes like MOCKCREDIT0000000001.
'''

print(SYSTEM_PROMPT)

---

### 6. 总结

构建一个准确的数据库 Agent，关键在于向模型传递领域知识：

**核心要点**：

- **Skill 的价值在于传递领域知识**——类型陷阱、业务规则、join 关系、枚举值
- **三处联动提示**——Schema 标注 + Example queries + Gotchas，确保模型不遗漏
- **description 是路由标签**——正面路由覆盖关键词，负面路由防止误用
- **Skill 是活文档**——每次答错就加一条提示，准确率逐步提升
- **System Prompt 只放工作流**——表结构知识放 Skill，保持 prompt 精简

**完整的目录结构**：

```
my_agent/
├── agent.yaml                        # Agent 配置
├── system_prompt.md                  # 工作流 + 全局约束
└── skills/
    ├── enterprise_basic/SKILL.md     # 企业基本信息
    ├── enterprise_contact/SKILL.md   # 联系人
    ├── enterprise_financing/SKILL.md # 融资与上市
    ├── enterprise_product/SKILL.md   # 产品与专利
    ├── industry/SKILL.md             # 行业链节点
    ├── industry_enterprise/SKILL.md  # 企业-行业映射
    └── users/SKILL.md                # 平台用户（负面路由）
```